In [24]:
from langchain.agents.middleware import SummarizationMiddleware
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver

In [25]:
memory_saver = MemorySaver()

In [26]:
llm = ChatOpenAI(model='gpt-4o-mini')

In [27]:
def print_agent_stats(agent, thread_id):
    config = {"configurable": {"thread_id": thread_id}}
    # Get the current state from the checkpointer
    state = agent.get_state(config)
    messages = state.values.get("messages", [])
    
    tokens = llm.get_num_tokens_from_messages(messages)
    
    print(f"\n--- Thread: {thread_id} ---")
    print(f"Message Count: {len(messages)}")
    print(f"Token Count:   {tokens}")
    for m in messages:
        # Show the type (Human, AI, or System summary)
        role = m.__class__.__name__.replace("Message", "")
        print(f"[{role}]: {m.content[:40]}...")
    print("-" * 25)

In [28]:
config = {"configurable": {"thread_id": "test000"}}

In [29]:
# Agent token limit
agent0 = create_agent(
    model='gpt-4o-mini',
    checkpointer= memory_saver,
 
)

In [30]:
result0 = agent0.invoke(
    {"messages": [{
        "role": "user",
        "content": "hi"
    }]},
    config=config
)

In [31]:
print_agent_stats(agent0, "test000")


--- Thread: test000 ---
Message Count: 2
Token Count:   21
[Human]: hi...
[AI]: Hello! How can I assist you today?...
-------------------------


In [32]:
result0 = agent0.invoke({
    "messages": [{
        'role': 'user', 'content': "tell me a story in 40  words"
    }]
}, config=config)

In [33]:
print_agent_stats(agent0, "test000")


--- Thread: test000 ---
Message Count: 4
Token Count:   88
[Human]: hi...
[AI]: Hello! How can I assist you today?...
[Human]: tell me a story in 40  words...
[AI]: In a forgotten village, a lonely owl dis...
-------------------------


In [34]:
result0 = agent0.invoke({
    "messages": [{
        'role': 'user', 'content': "hi"
    }]
}, config=config)

In [35]:
print_agent_stats(agent0, "test000")


--- Thread: test000 ---
Message Count: 6
Token Count:   107
[Human]: hi...
[AI]: Hello! How can I assist you today?...
[Human]: tell me a story in 40  words...
[AI]: In a forgotten village, a lonely owl dis...
[Human]: hi...
[AI]: Hi again! How can I help you today?...
-------------------------


In [36]:
# Agent token limit
agent1 = create_agent(
    model='gpt-4o-mini',
    checkpointer= memory_saver,
    middleware=[
        SummarizationMiddleware(
            model='gpt-4o-mini', 
            trigger=("tokens", 100),
            keep=("messages", 2)
        )
    ]
)

In [37]:
config = {"configurable": {"thread_id": "test001"}}

In [38]:
result1 = agent1.invoke(
    {"messages": [{
        "role": "user",
        "content": "hi"
    }]},
    config=config
)

In [39]:
print_agent_stats(agent1, "test001")


--- Thread: test001 ---
Message Count: 2
Token Count:   21
[Human]: hi...
[AI]: Hello! How can I assist you today?...
-------------------------


In [40]:
result1 = agent1.invoke({
    "messages": [{
        'role': 'user', 'content': "tell me a story in 40  words"
    }]
}, config=config)

In [41]:
print_agent_stats(agent1, "test001")


--- Thread: test001 ---
Message Count: 4
Token Count:   88
[Human]: hi...
[AI]: Hello! How can I assist you today?...
[Human]: tell me a story in 40  words...
[AI]: In a forgotten village, a young girl dis...
-------------------------


In [42]:
result1 = agent1.invoke({
    "messages": [{
        'role': 'user', 'content': "hi"
    }]
}, config=config)

In [43]:
print_agent_stats(agent1, "test001")


--- Thread: test001 ---
Message Count: 4
Token Count:   163
[Human]: Here is a summary of the conversation to...
[AI]: In a forgotten village, a young girl dis...
[Human]: hi...
[AI]: Hello! How can I assist you today?...
-------------------------


In [44]:
result1 = agent1.invoke({
    "messages": [{
        'role': 'user', 'content': "add 2 + 2"
    }]
}, config=config)

In [45]:
print_agent_stats(agent1, "test001")


--- Thread: test001 ---
Message Count: 4
Token Count:   153
[Human]: Here is a summary of the conversation to...
[AI]: Hello! How can I assist you today?...
[Human]: add 2 + 2...
[AI]: 2 + 2 equals 4....
-------------------------
